In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from PIL import Image
import json
from icecream import ic

ModuleNotFoundError: No module named 'icecream'

In [ ]:
# df = pd.read_pickle("VBERT_result.pkl")

In [67]:
i = 0
df.head()

,question_id,image_id,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer,visualBERT_predictions_with_visual,visualBERT_predictions_without_visual,visualBERT_scores_with_visual,visualBERT_scores_without_visual,visualBERT_prediction_logits
0,22jbM6gDxdaMaunuzgrsBB,461751,What is in the motorcyclist's mouth?,"[toothpick, food, popsicle stick, cigarette]",cigarette,3,He's smoking while riding. The motorcyclist ha...,cigarette\tcigarette\tcigarette\tcigarette\tci...,nothing,nothing,0.0,0.0,[tensor(1881)]
1,2Aq5RiEn7eyfWjEbpuYT2o,377368,Which number birthday is probably being celebr...,"[one, ten, nine, thirty]",thirty,3,There is a birthday cake on the table with the...,thirty\t30th\tthirty\tthirty\tthirty\t30th\tth...,2,unknown,0.0,0.0,[tensor(95)]
2,2Br4bJfKY7SQM9DECrqqeG,563603,What best describes the pool of water?,"[frozen, fresh, dirty, boiling]",dirty,2,The pool is dark brown. It it brown and surrou...,muddy\tdirty\tmurky water\tmuddy\tpond\tpond\t...,calm,glass,0.0,0.0,[tensor(648)]
3,2C8riXpRLX3CyM5jDz23m7,329542,What is the white substance on top of the cupc...,"[butter, mayo, ice cream, icing]",icing,3,This is frosting used to decorate and add more...,icing\twhipped cream\ticing\tfrosting\ticing\t...,sugar,nothing,0.0,0.0,[tensor(2660)]
4,2DQex53EkNGH2cfo3WPuPn,182202,What type of device is sitting next to the lap...,"[mouse, mobile phone, pen, keyboard]",mobile phone,1,It has the name of it on the top The device ha...,cell phone\tvodafone\tphone\tphone\tphone\tpho...,phone,laptop,1.0,0.0,[tensor(2087)]


In [7]:
qa_data = 0
def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco2017/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

from tqdm import tqdm# Process each sample in qa_data and compute predictions

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset
val_dataset = load_aokvqa(aokvqa_dir, 'val')

In [41]:

def get_qa_result(fname):
    with open('results\\' + fname, 'r') as file:
        result_data = json.load(file)

    qa_data = dict()
    for sample in val_dataset:
        question_id = sample["question_id"]
        image_id = sample["image_id"]
        question = sample["question"]
        choices = sample["choices"]
        correct_choice_idx = sample["correct_choice_idx"]
        rationales = sample.get("rationales", [])
        combined_rationale = " ".join(rationales)
        direct_answers = sample.get("direct_answers", [])
        combined_direct_answer = "\t".join(direct_answers)
        correct_ans = choices[correct_choice_idx]
        qa_data[question_id] = {
            "question_id": question_id,
            "image_id": image_id,
            "question": question,
            "choices": choices,
            "correct_answer": correct_ans,
            "correct_choice_idx": correct_choice_idx,
            "rationale": combined_rationale,
            "direct_answer": combined_direct_answer
        }
    for sample in result_data:
        question_id = sample["question_id"]
        current_dict = qa_data[question_id]
        current_dict["mc_answer"] = sample['mc_answer']
        current_dict["da_answer"] = sample['da_answer']
        current_dict["da_correctness"] = (sample['da_correctness'].strip().lower() == "correct")
        current_dict["mc_correctness"] = (sample["mc_answer"].strip().lower() == sample["correct_mc_answer"].strip().lower())
        qa_data[question_id] = current_dict

    qa_list = [qa_data[id] for id in qa_data]
    qa_df = pd.DataFrame(qa_list)

    return qa_data, qa_df

# print(f"VBERT Score with visual: {qa_df['visualBERT_scores_with_visual'].mean():.2%}")

In [42]:
qa_data, qa_df = get_qa_result("gpt4o_results_with_correctness.json")
ic(qa_df["da_correctness"].mean())
ic(qa_df["mc_correctness"].mean())
pass

ic| qa_df["da_correctness"].mean(): np.float64(0.3615720524017467)
ic| qa_df["mc_correctness"].mean(): np.float64(0.5458515283842795)


In [43]:
qa_data, qa_df = get_qa_result("cross_attention_results_with_correctness.json")
ic(qa_df["da_correctness"].mean())
ic(qa_df["mc_correctness"].mean())
pass

ic| qa_df["da_correctness"].mean(): np.float64(0.3685589519650655)
ic| qa_df["mc_correctness"].mean(): np.float64(0.5336244541484716)


In [45]:
qa_data, cot_qa_df = get_qa_result("cross_attention_cot_results_with_correctness.json")
ic(cot_qa_df["da_correctness"].mean())
ic(cot_qa_df["mc_correctness"].mean())
pass

ic| cot_qa_df["da_correctness"].mean(): np.float64(0.3720524017467249)


ic| cot_qa_df["mc_correctness"].mean(): np.float64(0.5353711790393013)


In [48]:
qa_data, cot_qa_df = get_qa_result("cross_attention_cot_6examples_results_with_correctness.json")
ic(cot_qa_df["da_correctness"].mean())
ic(cot_qa_df["mc_correctness"].mean())
pass

ic| cot_qa_df["da_correctness"].mean(): np

.float64(0.36681222707423583)
ic| cot_qa_df["mc_correctness"].mean(): np.float64(0.5240174672489083)


In [31]:
(cot_qa_df["mc_correctness"] == qa_df["mc_correctness"]).mean()

np.float64(0.8637554585152838)

In [40]:
import json

def compute_accuracies(results):
    """
    Compute MC accuracy and DA accuracy.
    results: list of dicts with keys 'mc_answer', 'correct_mc_answer', 'da_answer', 'correct_da_answer'
    For DA, correct_da_answer may be a list of reference strings.
    """
    num = len(results)
    mc_correct = 0
    da_correct = 0

    for r in results:
        # MC: case-insensitive match
        pred_mc = r.get("mc_answer", "").strip().lower()
        gold_mc = str(r.get("correct_mc_answer", "")).strip().lower()
        if pred_mc == gold_mc:
            mc_correct += 1

        # DA: use da_correctness flag if available, otherwise fallback to text match
        da_flag = r.get("da_correctness")
        if isinstance(da_flag, str):
            if da_flag.strip().lower() == "correct":
                da_correct += 1

    mc_acc = mc_correct / num * 100 if num > 0 else 0.0
    da_acc = da_correct / num * 100 if num > 0 else 0.0
    return mc_correct, mc_acc, da_correct, da_acc

if __name__ == "__main__":
    # List of JSON result files to evaluate
    json_files = [
        "cross_attention_cot_results_with_correctness.json",
        "cross_attention_results_with_correctness.json",
        "gpt4o_results_with_correctness.json"
    ]

    for path in json_files:
        try:
            with open("results\\" +path, "r") as f:
                results = json.load( f)
        except FileNotFoundError:
            print(f"File not found: {path}")
            continue

        mc_corr, mc_acc, da_corr, da_acc = compute_accuracies(results)
        total = len(results)
        print(f"Results for {path}:")
        print(f"  Total samples: {total}")
        print(f"  MC Accuracy: {mc_corr}/{total} = {mc_acc:.2f}%")
        print(f"  DA Accuracy: {da_corr}/{total} = {da_acc:.2f}%")
        print()

Results for cross_attention_cot_results_with_correctness.json:
  Total samples: 1145
  MC Accuracy: 613/1145 = 53.54%
  DA Accuracy: 426/1145 = 37.21%

Results for cross_attention_results_with_correctness.json:
  Total samples: 1145
  MC Accuracy: 611/1145 = 53.36%
  DA Accuracy: 422/1145 = 36.86%

Results for gpt4o_results_with_correctness.json:
  Total samples: 1145
  MC Accuracy: 625/1145 = 54.59%
  DA Accuracy: 414/1145 = 36.16%



In [ ]:
row = qa_df.iloc[i]
print(i)
print(row)
img_id = row['image_id']
img_path =  os.path.join("..", "datasets", "coco", "val2017", "img", f"{img_id:012}.jpg")
# img_path = os.path.join("..", "000000563603.jpg")
img = mpimg.imread(img_path)  # Load the image
plt.imshow(img)  # Show the image
plt.axis("off")  # Hide axes
plt.show()
i += 1


NameError: name 'qa_df' is not defined